## Feature Selection สำหรับการเลือกหุ้นเข้าพอร์ต (NASDAQ-100)
- แนวคิด: ทุกสิ้นเดือน สร้าง feature ของหุ้นแต่ละตัวจากข้อมูลย้อนหลัง 252 วัน แล้วดูว่า feature ตัวไหนบอกได้ว่า "หุ้นตัวนี้จะมี Sharpe ดีกว่าค่ากลางของตลาด" ใน 63 วันข้างหน้า

In [67]:
import pandas as pd
import numpy as np

## โหลดข้อมูล

In [68]:
df = pd.read_csv('NASDAQ-100/NASDAQ_100_Data_From_2010.csv', sep='\t')
df['Date'] = pd.to_datetime(df['Date'])
df = df.sort_values(['Name', 'Date']).reset_index(drop=True)

 ## สร้าง Feature ย้อนหลัง
 ### วิธีทำ
 1. สร้าง Features ย้อนหลัง
 2. สร้าง Target Forward 63-Day Sharpe > Median ของเดือนนั้นๆ
 3. กรองเฉพาะข้อมูลวันสิ้นเดือนเพื่อลด Overlap

In [69]:
# step 1
df['Daily_Return'] = df.groupby('Name')['Adj Close'].pct_change()
df['Ret_252d'] = df.groupby('Name')['Adj Close'].pct_change(252)
df['Ret_63d'] = df.groupby('Name')['Adj Close'].pct_change(63)
df['Vol_252d'] = df.groupby('Name')['Daily_Return'].transform(lambda x: x.rolling(252).std() * np.sqrt(252))
df['Sharpe_252d'] = df['Ret_252d'] / (df['Vol_252d'] + 1e-6)

df['MA_50'] = df.groupby('Name')['Adj Close'].transform(lambda x: x.rolling(50).mean())
df['MA_200'] = df.groupby('Name')['Adj Close'].transform(lambda x: x.rolling(200).mean())
df['MA_Ratio'] = df['MA_50'] / (df['MA_200'] + 1e-6)

# step 2
def calc_fwd_vol(group):
    return group['Daily_Return'].iloc[::-1].rolling(63).std().iloc[::-1] * np.sqrt(252)

df['Fwd_Ret_63d'] = df.groupby('Name')['Adj Close'].shift(-63) / df['Adj Close'] - 1
df['Fwd_Vol_63d'] = df.groupby('Name', group_keys=False).apply(calc_fwd_vol)
df['Fwd_Sharpe'] = df['Fwd_Ret_63d'] / (df['Fwd_Vol_63d'] + 1e-6)

# step 3
df['YearMonth'] = df['Date'].dt.to_period('M')
monthly_df = df.groupby(['Name', 'YearMonth']).tail(1).reset_index()
monthly_df['Target'] = monthly_df.groupby('YearMonth')['Fwd_Sharpe'].transform(lambda x: (x.gt(x.median()).where(x.notna()).astype(float)))

feature_cols = ['Ret_252d', 'Ret_63d', 'Vol_252d', 'Sharpe_252d', 'MA_Ratio']
clean_df = monthly_df.dropna(subset=feature_cols + ['Target'].copy())

## เปรียบเทียบ Featureselection ทุกรูปแบบ
### Standardize Features

In [70]:
from sklearn.preprocessing import StandardScaler

x = clean_df[feature_cols]
y = clean_df['Target']

scaler = StandardScaler()
X_scaled = pd.DataFrame(scaler.fit_transform(x), columns=feature_cols)

results = pd.DataFrame(index=feature_cols)
results

""
Ret_252d
Ret_63d
Vol_252d
Sharpe_252d
MA_Ratio


## Filter Method: Mutual Information
- วิธีการดู: ดูค่าคะแนน ยิ่งสูงยิ่งดี
- ความหมาย: Feature ตัวนั้นมีข้อมูลร่วมหรือมีความสัมพันธ์กับ Target มากแค่ไหน
- วิธีเลือก: เรียงลำดับจากมากไปน้อย แล้วเลือก Top $N$ ตัวแรกที่มีค่าสูงสุด

In [71]:
from sklearn.feature_selection import mutual_info_classif

results['Mutual_Info'] = mutual_info_classif(X_scaled, y, random_state=42)

results

,Mutual_Info
Ret_252d,0.005023
Ret_63d,0.011351
Vol_252d,0.001423
Sharpe_252d,0.000717
MA_Ratio,0.000000


## Embedded Method: L1 Regularization Lasso Logistic Regression
- วิธีดู: ดูขนาดของสัมปสิทธิ์ ยิ่งไกลจาก 0 ยิ่งดี
- L1 Regularization จะบีบให้ Feature ที่ไม่มีประโยชน์มีค่า Coefficient เป็น 0 โดยอัตโนมัติ
- วิธีเลือก:
  - แบบสตรีค: เลือกเฉพาะ Feature ที่มีค่า L1_Coef_Abs > 0
  - แบบจัดอันดับ: เลือกตัวที่มีค่า Absolute Coefficient สูงที่สุด $N$ อันดับแรก

In [77]:
from sklearn.linear_model import LogisticRegression

log_l1 = LogisticRegression(penalty='l1', solver='liblinear', random_state=42)
log_l1.fit(X_scaled, y)
results['L1_Coef_Abs'] = np.abs(log_l1.coef_[0])

results

C:\Users\piyaw\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\linear_model\_logistic.py:1381: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(
C:\Users\piyaw\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\linear_model\_logistic.py:1407: UserWarning: Inconsistent values: penalty=l1 with l1_ratio=0.0. penalty is deprecated. Please use l1_ratio only.
  warnings.warn(


,Mutual_Info,RF_Gini_Importance,Permutation_Importance,RFE_Rank,L1_Coef_Abs
Ret_252d,0.005023,0.192861,0.162365,3,0.040735
Ret_63d,0.011351,0.204215,0.219984,2,0.029871
Vol_252d,0.001423,0.203122,0.224739,1,0.060085
Sharpe_252d,0.000717,0.193846,0.175480,1,0.034180
MA_Ratio,0.000000,0.205956,0.231898,1,0.060932


## Embedded Method: Random Forest Feature Importance
- วิธีดู: ดูค่าคะแนน ยิ่งสูงยิ่งดี (ผลรวมทุก Feature รวมกันจะเท่ากับ 1.0 หรือ 100%)
- ความหมาย: วัดว่า Feature ตัวนั้นช่วยลดความไม่บริสุทธิ์ (Impurity / Gini) ในการแบ่ง Node ของ Tree ได้มากแค่ไหน
- วิธีเลือก: เรียงลำดับจากมากไปน้อย แล้วเลือก Feature ที่ได้ Percent Share สูงสุด $N$ อันดับแรก

In [73]:
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X_scaled, y)
results['RF_Gini_Importance'] = rf.feature_importances_

results

,Mutual_Info,RF_Gini_Importance
Ret_252d,0.005023,0.192861
Ret_63d,0.011351,0.204215
Vol_252d,0.001423,0.203122
Sharpe_252d,0.000717,0.193846
MA_Ratio,0.000000,0.205956


## Embedded Method: Permutation Importance
- วิธีดู: ดูค่าคะแนน ยิ่งสูงยิ่งดี (ถ้าค่าติดลบหรือเข้าใกล้ 0 แสดงว่าเป็น Noise)
- ความหมาย: ลองสลับสับเปลี่ยนข้อมูล (Shuffle) ของ Feature นั้นๆ แล้วดูว่าความแม่นยำของโมเดลลดลงเท่าไหร่ ยิ่งความแม่นยำร่วงหนัก แปลว่า Feature นั้นสำคัญมาก
- วิธีเลือก: เลือก Feature ที่มีค่า Permutation_Importance > 0 และเรียงลำดับเลือก $N$ ตัวแรกที่มีค่าสูงสุด

In [74]:
from sklearn.inspection import permutation_importance

perm_imp = permutation_importance(rf, X_scaled, y, n_repeats=10, random_state=42)
results['Permutation_Importance'] = perm_imp.importances_mean

results

,Mutual_Info,RF_Gini_Importance,Permutation_Importance
Ret_252d,0.005023,0.192861,0.162365
Ret_63d,0.011351,0.204215,0.219984
Vol_252d,0.001423,0.203122,0.224739
Sharpe_252d,0.000717,0.193846,0.175480
MA_Ratio,0.000000,0.205956,0.231898


## Wrapper Method: Recursive Feature Elimination
- วิธีดู: ดูค่า Rank = 1 (ตัวเลขยิ่งน้อยยิ่งดี)
- ความหมาย: อัลกอริทึมจะค่อยๆ ตัด Feature ที่แย่ที่สุดออกทีละตัวจนเหลือตามจำนวน n_features_to_select ที่เราตั้งไว้
- วิธีเลือก: เลือก Feature ที่มีค่า RFE_Rank == 1 เท่านั้น (เพราะ Rank 1 คือกลุ่มที่ผ่านการคัดเลือกเข้ามา)

In [75]:
from sklearn.feature_selection import RFE

rfe = RFE(estimator=rf, n_features_to_select=3)
rfe.fit(X_scaled, y)
results['RFE_Rank'] = rfe.ranking_

results

,Mutual_Info,RF_Gini_Importance,Permutation_Importance,RFE_Rank
Ret_252d,0.005023,0.192861,0.162365,3
Ret_63d,0.011351,0.204215,0.219984,2
Vol_252d,0.001423,0.203122,0.224739,1
Sharpe_252d,0.000717,0.193846,0.175480,1
MA_Ratio,0.000000,0.205956,0.231898,1


In [76]:
# เลือกเฉพาะ Top Features ที่ผ่านการคัดเลือก
selected_features = ['Vol_252d', 'MA_Ratio', 'Ret_252d']

X_selected = X_scaled[selected_features]

# เทรนโมเดลด้วยเฉพาะ Selected Features
final_model = RandomForestClassifier(n_estimators=100, random_state=42)
final_model.fit(X_selected, y)

# ทำนายความน่าจะเป็นของหุ้นในเดือนล่าสุด เพื่อเลือก Top 20 หุ้นนำไปจัดพอร์ต
latest_month = clean_df['YearMonth'].max()
latest_data = clean_df[clean_df['YearMonth'] == latest_month].copy()

latest_X_scaled = scaler.transform(latest_data[feature_cols])
latest_X_selected = pd.DataFrame(latest_X_scaled, columns=feature_cols)[selected_features]

# ทำนายโอกาสที่จะได้ Forward Sharpe สวยๆ (Probability of Target = 1)
latest_data['Prob_Success'] = final_model.predict_proba(latest_X_selected)[:, 1]

# ดึง Top 20 หุ้นดาวเด่น
top_20_stocks = latest_data.sort_values('Prob_Success', ascending=False).head(20)['Name'].tolist()

print("หุ้น Top 20 ตัวที่คัดเลือกผ่าน ML Pipeline:")
print(top_20_stocks)

หุ้น Top 20 ตัวที่คัดเลือกผ่าน ML Pipeline:
['ADSK', 'AAPL', 'EBAY', 'AMD', 'ADP', 'GOOG', 'INTU', 'LULU', 'GOOGL', 'MELI', 'MRNA', 'QCOM', 'CTSH', 'CHKP', 'TEAM', 'ANSS', 'VRSK', 'FAST', 'OKTA', 'CPRT']
